ARCHS4 Project
**************

Notes:
- Previously used Pickle Files, still use them but sometimes used .txt for easier flexability between Python and R

# 1. SAMPLE SELECTION 
    - BIOSAMPLE-CATEGORY SELECTION
    - Cell lines were manually combined into the "Tissue Types" Body Systems (ie. Brain cell lines were placed into "Nervous System")

In [4]:
%store -z

In [5]:
%%time
from itertools import islice
from collections import Counter
import numpy as np
import h5py
import re

# Open ARCHS4 v2.1 Data 
infile = "/home/ajoshi/archs4/data/archs4_gene_human_v2.1.h5"

# - Use read_direct method to improve read performance
with h5py.File(infile, 'r') as hdf5_file:
    data = hdf5_file['data']
    expression = data['expression'][:]

    meta = hdf5_file['meta']
    symbols = meta['genes']['gene_symbol'][:]
    source_name_ch1 = meta['samples']['source_name_ch1'][:]

# Decode source names using list comprehension
source_decoded = [x.decode('UTF-8') for x in source_name_ch1]

# Clean source names using regular expressions
source_cleaned = [re.sub(r'[-/\'\\ _]', '', x) for x in source_decoded]

# List of all categories (keywords)
keyword_list = [# Cardiovascular 
                "Heart", "Atrium", "Cardiac Muscle Fiber", "Pericardium", "Valve", "Ventricle",
                "HUVEC", # Endothelial 
                
                # Connective Tissue
                "Adipose", "Adipocyte",
                "Myofibroblast", "Osteoblast", "Stromal Cell", "MG63", "K562", # Bone
                "Bone Marrow", 
                "Chondrocyte", 
                "HT1080", "NHDF", # Connective
                
                # Digestive
                "Esophagus",
                "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", "Small Intestine", 
                "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell", "HEPG2", # Liver
                "Alpha Cell", "Beta Cell", "Pancreatic Islet", "PANC1", # Pancreas
                "Gastric Epithelial Cell", 
                "HCT116", "HT29", "RKO", # Connective, Colon
                
                # Immune
                "Granulocyte", "Neutrophil",
                "Blymphocyte", "Plasma Cell", "Plasmacytoid Dendritic Cell", "Tlymphocyte", 
                "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", # Lymphoid
                "Alveolar", "Dendritic", "Kupffer", "Macrophage", "Macroglia", 
                "THP1", "U937", # Myeloid
                "JEG3", # Placenta
                "Spleen", 
                "Thymus", "Thymocyte", 
                
                # Integumentary
                "Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte", 
                "A431", "HACAT", "HNSCC",
                
                # Muscular
                "Skeletal Muscle", "Myoblast", 
                "Airway Smooth Muscle", #"Myofibroblast", [Repeat]
                "Respiratory Smooth Muscle", "Vascular Smooth Muscle", 
                "LHCNM2", # Muscle
                
                # Nervous
                "Astrocyte", "Cerebellum", "Cerebral Cortex", "Hypothalamus", "Medula", 
                "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", "Thalamus", 
                "Retina", 
                "Motor Neuron", "Sensory Neuron",
                "Brain", # Example in the ARCHS4 Website [1]
                "DAOY", "IMR32", "SKNSH", # Brain 
                
                # Respiratory 
                #"Alveolar", [Repeat] 
                "Lung", "Lung Epithelial", 
                "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", # Lung
                "Trachea", 
                
                # Urogenital 
                "Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D",
                "HELA", # Cervix
                "Kidney", "Podocyte", "Renal Cortex",
                "293F", "FLPIN TREX 293", "HEK293", # Kidne
                "C42", "DU145", "LNCAP", "PC3", # Prosta
                
                "Ovary", "Granulosa", "Oocyte",
                "ES2", "SKOV3", # Ovary
                "Testis", 
                "GSE81547", # Example in the ARCHS4 GitHub
                "GSE30017",  "GSM2679484"] # Examples in the ARCHS4 Website [2]

# Create a dictionary that maps keywords to their indices
category_dict = {keyword: [idx for idx, string in enumerate(source_cleaned) 
                           if keyword.replace(" ", "").lower() in string.lower()] for keyword in keyword_list}

# Get the total sample size collected for each keyword 
category_sample_sizes = {key: len(category_dict[key]) for key in category_dict}


MemoryError: Unable to allocate 145. GiB for an array with shape (62548, 620825) and data type uint32

In [3]:
# DATA CHECK #
##############

print("Check if first and last element in heart category contains the 'heart' keyword")
print(source_cleaned[5261], "\n", source_cleaned[620354], "\n")

total_keys = len(category_dict)
print("Total Keys:", total_keys)

total_samples = sum(category_sample_sizes.values())
print("Total Samples:", total_samples)  

#print("Category Sample Sizes:", category_sample_sizes)

# print("category_dict - First 3 elements:")
# for key, value in islice(category_dict.items(), 3):
#     print(key, value)

# SAMPLE OVERLAPPING BT CATEGORIES FREQUENCY COUNTER CHECK #
############################################################

# Flatten the list values into a single list of integers
allSelected_sampleIndices_list = [item for sublist in category_dict.values() for item in sublist]
print("allSampleIndices_list size [Should be 203k]:", len(allSampleIndices_list))

# Create a frequency dictionary using Counter
freqSampleIndices_dict = Counter(allSelected_sampleIndices_list)

# Find the total number of unique values in the flattened list
unique_total_samples = len(freqSampleIndices_dict)

print("Total Unique Samples:", unique_total_samples)

Check if first and last element in heart category contains the 'heart' keyword


NameError: name 'source_cleaned' is not defined

View Jupyter Notebook Store Variables 

In [ ]:
%whos

In [ ]:
# STORE DATA #
##############

#%store expression
%store category_dict
%store category_sample_sizes

# Save backup dictionary to txt
with open('/home/lwu/ARCHS4_v2-1/Data Results/category_sampleIndices_dict_LW01082023.txt','w') as data: 
      data.write(str(category_dict))

with open('/home/lwu/ARCHS4_v2-1/Data Results/category_totalSampleSize_dict_LW01082023.txt','w') as data: 
      data.write(str(category_sample_sizes))

In [ ]:
{'Heart': 709,
 'Atrium': 67,
 'Cardiac Muscle Fiber': 0,
 'Pericardium': 10,
 'Valve': 136,
 'Ventricle': 769,
 'HUVEC': 1178,
 'Adipose': 3546,
 'Adipocyte': 468,
 'Myofibroblast': 66,
 'Osteoblast': 255,
 'Stromal Cell': 730,
 'MG63': 30,
 'K562': 5077,
 'Bone Marrow': 11303,
 'Chondrocyte': 166,
 'HT1080': 78,
 'NHDF': 68,
 'Esophagus': 1308,
 'Colon': 4083,
 'Colonic Muscosa': 0,
 'Intestinal Epithelial Cell': 55,
 'Small Intestine': 411,
 'Liver': 9438,
 'Hepatic Stellate Cell': 130,
 'Hepatocyte': 2171,
 'Primary Liver Cell': 2,
 'HEPG2': 2383,
 'Alpha Cell': 146,
 'Beta Cell': 241,
 'Pancreatic Islet': 14000,
 'PANC1': 352,
 'Gastric Epithelial Cell': 12,
 'HCT116': 1826,
 'HT29': 416,
 'RKO': 143,
 'Granulocyte': 91,
 'Neutrophil': 947,
 'Blymphocyte': 88,
 'Plasma Cell': 832,
 'Plasmacytoid Dendritic Cell': 153,
 'Tlymphocyte': 215,
 'JURKAT': 1569,
 'MT4': 10,
 'NALM6': 846,
 'REH': 180,
 'RPMI8226': 74,
 'Alveolar': 588,
 'Dendritic': 1602,
 'Kupffer': 31,
 'Macrophage': 4084,
 'Macroglia': 0,
 'THP1': 1449,
 'U937': 263,
 'JEG3': 22,
 'Spleen': 2385,
 'Thymus': 145,
 'Thymocyte': 11,
 'Skin': 10947,
 'Basal Cell': 447,
 'Fibroblast': 12418,
 'Hair Follicle': 67,
 'Keratinocyte': 1010,
 'Melanocyte': 158,
 'A431': 38,
 'HACAT': 151,
 'HNSCC': 216,
 'Skeletal Muscle': 679,
 'Myoblast': 1149,
 'Airway Smooth Muscle': 157,
 'Respiratory Smooth Muscle': 0,
 'Vascular Smooth Muscle': 89,
 'LHCNM2': 26,
 'Brain': 12603,
 'DAOY': 75,
 'IMR32': 291,
 'SKNSH': 93,
 'Astrocyte': 805,
 'Cerebellum': 534,
 'Cerebral Cortex': 72,
 'Hypothalamus': 23,
 'Medula': 0,
 'Midbrain': 2128,
 'Neuron': 7760,
 'Oligodendrocyte': 327,
 'Pons': 736,
 'Spinal Cord': 588,
 'Thalamus': 27,
 'Retina': 9188,
 'Motor Neuron': 448,
 'Sensory Neuron': 62,
 'Lung': 10975,
 'Lung Epithelial': 222,
 'A549': 2139,
 'H1299': 330,
 'IMR90': 3046,
 'MRC5': 199,
 'NCIH460': 15,
 'NHBE': 90,
 'Trachea': 111,
 'Breast': 12250,
 'Mammary Gland': 246,
 'MCF7': 6015,
 'MCF10': 1209,
 'MDAMB231': 1328,
 'MDAMB468': 277,
 'SKBR3': 149,
 'T47D': 624,
 'HELA': 3660,
 'Kidney': 6643,
 'Podocyte': 123,
 'Renal Cortex': 50,
 '293F': 479,
 'FLPIN TREX 293': 113,
 'HEK293': 6745,
 'C42': 311,
 'DU145': 387,
 'LNCAP': 2115,
 'PC3': 652,
 'Ovary': 498,
 'Granulosa': 199,
 'Oocyte': 380,
 'ES2': 739,
 'SKOV3': 174,
 'Testis': 715,
 'GSE81547': 0,
 'GSE30017': 0,
 'GSM2679484': 0}

# 2. READ COUNT SELECTION 
    - Top 100k

In [ ]:
%%time 

# Unable to use dict, since sample indices overlap (duplicates)
sampleIndx_totalReadCount_list = []

# Get sample totalReadCount list
readCount_list = np.sum(expression, axis = 0)

# Calculate each sample indice read count [corrsponding to: allSampleIndices_list]
for sample_indx in allSelected_sampleIndices_list:
        sampleIndx_totalReadCount_list.append(readCount_list[sample_indx])

In [ ]:
# DATA CHECK #
##############

print("Total readCount_list [Should match 620825 samples (rows)]: ", len(readCount_list))

print()
print("Total # Samples: [Should be 200k]:", len(sampleIndx_totalReadCount_list))
print()
print("First three elements: ")
for key, value in islice(sampleIdx_totalReadCount_dict.items(), 3):
    print(key, value)
    


In [ ]:
# Reorganize the sample indices based on descending total read counts 

# Zip the <sample indices> & <sample indices' read count> together
zipped_lists = zip(allSelected_sampleIndices_list, sampleIndx_totalReadCount_list)

# Sort the list of tuples in descending order of the <sample indices' read count> (second list)
sorted_lists = sorted(zipped_lists, key=lambda x: x[1], reverse=True)

# Unzip the sorted list of tuples back into two lists (first list is the important one)
ordered_selected_sampleIndices_list, ordered_sampleIndx_totalReadCount_list = zip(*sorted_lists)

print("ordered_selected_sampleIndices_list size (should be 200k):", len(ordered_selected_sampleIndices_list))
print("First 5 Highest Read Count Samples:", ordered_selected_sampleIndices_list[:5])  
print("First 5 Highest Read Counts:", ordered_sampleIndx_totalReadCount_list[:5])  



In [ ]:
# DATA CHECK - CUROSITY #
#########################

source_cleaned[347736]

# Highest Read Count Sample Overlaps Three Categories 
# Lung: A549, Breast/Mammary: MCF7, Bone: K562

In [ ]:
# Select the top 100k samples based on highest read counts

# Get the top 100k sample indxs (keys) with the highest values
top_100k_samples = ordered_selected_sampleIndices_list[:100000]

print("Check top_100k_samples size is 100k:", len(top_100k_samples))

In [ ]:
# STORE DATA #
##############

%store readCount_list
#%ordered_selected_sampleIndices_list # ERROR: Unable to be found???
#%ordered_sampleIndx_totalReadCount_list
%store top_100k_samples

# Save backup dictionary to txt
with open('/home/lwu/ARCHS4_v2-1/Data Results/entireARCHS4_readCount_list_LW01082023.txt','w') as data: 
      data.write(str(readCount_list))
        
with open('/home/lwu/ARCHS4_v2-1/Data Results/ordered_selected_sampleIndices_list_LW01082023.txt','w') as data: 
      data.write(str(ordered_selected_sampleIndices_list))
        
with open('/home/lwu/ARCHS4_v2-1/Data Results/ordered_sampleIndx_totalReadCount_list_LW01082023.txt','w') as data: 
      data.write(str(ordered_sampleIndx_totalReadCount_list))

with open('/home/lwu/ARCHS4_v2-1/Data Results/top_100k_samples_LW01082023.txt','w') as data: 
      data.write(str(top_100k_samples))

READ COUNT SELECTION NOTES 
**************************

Total # Samples Selected: 100,000
Total # Samples Removed: 103,878

# 3. ANALYZE REMOVED BIOSAMPLES 

In [ ]:
removed_103k_keys = ordered_selected_sampleIndices_list[100000:]

In [ ]:
# WARNING NOTE: This includes duplicates sample indices

selected_sample_names = []

for sample_idx in top_100k_samples:
    
    selected_sample_names.append(source_cleaned[sample_idx])
    
# Create a dictionary that maps keywords to their indices
selectedSamples_categories_dict = {keyword: [idx for idx, string in enumerate(selected_sample_names) 
                           if keyword.replace(" ", "").lower() in string.lower()] for keyword in keyword_list}

# Get the total sample size collected for each keyword 
selectedSamples_category_sizes = {key: len(selectedSamples_categories_dict[key]) for key in selectedSamples_categories_dict}


In [ ]:
selectedSamples_category_sizes

In [ ]:
unique_top_100k_samples = set(top_100k_samples)
    
unique_selected_sampleNames = []

for sample_idx in unique_top_100k_samples:
    
    unique_selected_sampleNames.append(source_cleaned[sample_idx])
    
# Create a dictionary that maps keywords to their indices
uniqueSelectedSamples_categories_dict = {keyword: [idx for idx, string in enumerate(unique_selected_sampleNames) 
                           if keyword.replace(" ", "").lower() in string.lower()] for keyword in keyword_list}

# Get the total sample size collected for each keyword 
uniqueSelectedSamples_category_sizes = {key: len(uniqueSelectedSamples_categories_dict[key]) 
                                        for key in uniqueSelectedSamples_categories_dict}


In [ ]:
uniqueSelectedSamples_category_sizes

3A. CHECK LOSS IN SAMPLES BETWEEN CATEGORIES AFTER READ COUNT SELECTION 

In [ ]:
# Get the sample size based on categories differences 
differences = [(key, category_sample_sizes[key] - uniqueSelectedSamples_category_sizes[key]) for key in category_sample_sizes]

# Sort the list in descending order 
sorted_differences = sorted(differences, key=lambda x: x[1], reverse=True)

# Iterate through the sorted list and print the key and the difference
for key, value in sorted_differences:
    print(key, value)

In [ ]:
# PLOT #
########

import matplotlib.pyplot as plt

# Create a list of keys and a list of values for dict1
original_archs4_category_sampleSizes = list(category_sample_sizes.values())

# Create a list of values for dict2
afterSelection_archs4_category_sampleSizes = list(uniqueSelectedSamples_category_sizes.values())

# Set the x-axis values to be the categories
x_axis = range(len(keyword_list))

# Create the line plot
plt.plot(x_axis, original_archs4_category_sampleSizes, label='Original ARCHS4 Biosamples')
plt.plot(x_axis, afterSelection_archs4_category_sampleSizes, label='After Read Count Filteration')

# Set the tick locations and labels
plt.xticks(x_axis, keyword_list, rotation=90, horizontalalignment='right', fontsize='x-small')
plt.tight_layout()
ax = plt.gca()
for label in ax.get_xaxis().get_ticklabels()[::2]:
    label.set_visible(False)

sorted_x_axis = [x for _, x in sorted(zip(original_archs4_category_sampleSizes, x_axis), key=lambda pair: pair[0], reverse=True)]
sorted_original_archs4_category_sampleSizes = sorted(original_archs4_category_sampleSizes, reverse=True)

# Create the line plot
plt.plot(sorted_x_axis, sorted_original_archs4_category_sampleSizes, label='Original ARCHS4 Biosamples')
# Add a legend and label the axes
plt.legend()
plt.xlabel('Category')
plt.ylabel('# Samples')

# Show the plot
plt.show()

In [ ]:
# PLOT #
########

import matplotlib.pyplot as plt

# Create a list of keys and a list of values for dict1
original_archs4_category_sampleSizes = list(category_sample_sizes.values())

# Create a list of values for dict2
afterSelection_archs4_category_sampleSizes = list(uniqueSelectedSamples_category_sizes.values())

# Set the x-axis values to be the categories
x_axis = range(len(keyword_list))

# Create the bar chart
plt.bar(x_axis, original_archs4_category_sampleSizes, label='Original ARCHS4 Biosamples')
plt.bar(x_axis, afterSelection_archs4_category_sampleSizes, label='After Read Count Filteration')

# Set the tick locations and labels
plt.xticks(x_axis, keyword_list, rotation=90, horizontalalignment='right', fontsize='x-small')
plt.tight_layout()
ax = plt.gca()
for label in ax.get_xaxis().get_ticklabels()[::2]:
    label.set_visible(False)
    
# Add a legend and label the axes
plt.legend()
plt.xlabel('Category')
plt.ylabel('# Samples')

# Show the plot
plt.show()

DATA DIFF NOTES
***************


# 3. Gene Length Correction 

In [ ]:
# Open and read the gene-length file
with open("/home/meuleman/projects/ARCH4_data/WM20221126_Tx_length_normalization/transcriptLengthPerGene_WM20221126.txt", "r") as f:
    lines = f.readlines()[:]

# Extract the data from the lines
genes_data = [line.strip().split("\t") for line in lines]

# Extract the data from the genes_data list and store them in separate lists
wm_genes_symbol, wm_genes_id, wm_gene_length = zip(*genes_data)

# Create a dictionary mapping gene symbols to gene lengths
wm_dict = {gene_symbol: gene_length for gene_symbol, _, gene_length in genes_data}

# Print the lengths of the lists and the min gene length
print("WM Gene File Length: ", len(genes_data))
print("First Three Gene Data:\n", genes_data[0:3])
print("Min Gene Length: ", min(wm_gene_length))


In [ ]:
# The 57k matched genes between the Gene-Length-File & g:Convert-csv

import pickle

pickle_in = open("/home/lwu/ARCHS4_v2-1/Data Results/matched_genes_bt_gConvertCsv_GLF.pkl", 'rb')
csv_glf_genes_dict = pickle.load(pickle_in)
pickle_in.close()

print("# Matched Genes:", len(csv_glf_genes_dict))

print()
print("Check First 3 Genes in CSV-GLF Dict:")
for key, value in islice(csv_glf_genes_dict.items(), 3):
    print(key, value)

In [ ]:
%%time
num_rows, num_cols = expression.shape
new_data = []
for gene_row in range(num_rows):
    
    # Gene index
    #print(gene_row)
    
    # Gene name 
    gene_name = list(csv_glf_genes_dict.keys())[gene_row]
    #print("gene_name:", gene_name)
    
    # Gene data
    gene_data = expression[gene_row]
    #print("row:", gene_data[0:5])
        
    # GLF Gene name 
    glf_gene_name = csv_glf_genes_dict[gene_name]
    #print("glf gene_name:", glf_gene_name)
    
    # Gene length 
    gene_length = wm_dict[glf_gene_name]
    #print("length", gene_length)
    
    #print(type(gene_data))
    #print(type(gene_length))
    # New row 
    new_gene_row = gene_data/int(gene_length)
    #print("new_row:", new_gene_row[0:5])
    test = new_gene_row.tolist()
    new_data.append(test)
    #break
    

In [ ]:
infile = "/home/lwu/ARCHS4/Biosamples_Selection/Samples_Filteration [IP]/extracted_data.npy"

extractedData_matrix = np.load(infile)

print("Extraced data matrix size (should be 100k):", len(extractedData_matrix))

In [ ]:
extractedData_matrix.shape

In [ ]:
extractedData_matrix[0]

4A. UNMATCHED GENES - WHAT ARE THEIR EXPRESSION LEVELS?
    - GET THE ROW SUM VALUES FOR READ COUNTS
    - ^ & FOR THE SELECTED BIOSAMPLES

# 4. QUARTILE NORMALIZATION 
    - TEST CASE OF 5 GENES FIRST
    - COMPARE WITH THE ARCHS4 LIBRARY (ACCURACY CHECK) [ERROR: UNABLE TO OPEN PACKAGE RN]

In [ ]:
import numpy as np


# Save the first 5 sample gene (rows) for testing
expression = extractedData_matrix[:5, :]
#expression = np.array(new_array)

def transform_chunk(chunk):
    # Sort the chunk along the rows
    chunk = np.sort(chunk, axis=0)

    # Compute the percentile ranks of the elements along the rows
    percentile_ranks = 100.0 * (np.arange(chunk.shape[0]) + 1) / (chunk.shape[0] + 1)

    # Reshape the percentile ranks into a column vector
    percentile_ranks = percentile_ranks.reshape((-1, 1))

    # Compute the quantiles by inverting the percentile ranks
    quantiles = np.percentile(chunk, percentile_ranks, axis=0)

    # Interpolate the quantiles to the original values
    chunk = np.interp(chunk, quantiles, percentile_ranks)

    return chunk

# Split the array into chunks
num_chunks = 8  # Set the number of chunks to 8
chunk_size = expression.shape[0] // num_chunks
chunks = [expression[i * chunk_size:(i + 1) * chunk_size, :] for i in range(num_chunks)]

# Transform each chunk separately
transformed_chunks = [transform_chunk(chunk) for chunk in chunks]

# Concatenate the transformed chunks back into a single array
expression = np.concatenate(transformed_chunks)





In [ ]:
expression

In [ ]:
from sklearn.preprocessing import quantile_transform

# Save the first 5 sample gene (rows) for testing
new_array = extractedData_matrix[:5, :]

print(new_array)

# Perform log transformation
expression = np.log2(expression + 1)

# Perform normalization
expression = quantile_transform(expression, n_quantiles=100, random_state=0, copy=True)

In [ ]:
# PARALLELIZATION #
###################

import concurrent.futures
import numpy as np
from sklearn.preprocessing import quantile_transform

# Define a function to perform the log transformation and normalization on a chunk of the matrix
def transform_chunk(chunk):
    chunk = np.log2(chunk + 1)
    chunk = quantile_transform(chunk, n_quantiles=100, random_state=0, copy=True)
    return chunk

# Split the matrix into chunks
num_chunks = 8  # Set the number of chunks to 8
chunk_size = expression.shape[0] // num_chunks
chunks = [expression[i * chunk_size:(i + 1) * chunk_size, :] for i in range(num_chunks)]

# Create a pool of worker threads or processes
with concurrent.futures.ProcessPoolExecutor() as executor:
    # Submit a task for each chunk to the executor
    tasks = [executor.submit(transform_chunk, chunk)]